# Space-Track CDM Data: Exploratory Data Analysis

This notebook explores the Conjunction Data Messages (CDMs) fetched from Space-Track's `cdm_public` class.
Unlike our ESA Kelvins dataset (historical, 2015-2019), these are **live, real-time CDMs** from the 18th Space
Defense Squadron covering the last ~30 days.

Key fields per CDM:
- **Pc (Probability of Collision)** — computed with SP ephemerides + full covariance
- **MIN_RNG (miss distance in km)** — predicted closest approach distance
- **TCA** — Time of Closest Approach
- **Object metadata** — NORAD IDs, names, types (PAYLOAD/DEBRIS/ROCKET BODY), RCS
- **Emergency reportable** — flagged by 18 SDS for high-risk events

Our local store: `data/prediction_logs/cdm_store.jsonl`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

CDM_STORE = Path('../data/prediction_logs/cdm_store.jsonl')
TLE_SNAPSHOT_DIR = Path('../data/tle_snapshots')

## 1. Data Overview

In [ ]:
# Load CDM store
records = []
with open(CDM_STORE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'Total CDM records: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f'\nSchema:')
df.info()

In [ ]:
# Parse dates
df['tca_dt'] = pd.to_datetime(df['tca'], errors='coerce')
df['created_dt'] = pd.to_datetime(df['creation_date'], errors='coerce')
df['fetched_dt'] = pd.to_datetime(df['fetched_at'], errors='coerce', utc=True)

print(f'TCA date range: {df["tca_dt"].min()} to {df["tca_dt"].max()}')
print(f'Creation date range: {df["created_dt"].min()} to {df["created_dt"].max()}')
print(f'Unique CDM IDs: {df["cdm_id"].nunique()}')
print(f'Unique SAT1 objects: {df["sat1_norad"].nunique()}')
print(f'Unique SAT2 objects: {df["sat2_norad"].nunique()}')
print(f'Unique object pairs: {df.groupby(["sat1_norad", "sat2_norad"]).ngroups}')

df.head(3)

## 2. Probability of Collision (Pc) Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log10(Pc) histogram
log_pc = np.log10(df['pc'].clip(lower=1e-20))
axes[0].hist(log_pc, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('log10(Pc)')
axes[0].set_ylabel('Count')
axes[0].set_title('Probability of Collision Distribution')
axes[0].axvline(x=np.log10(1e-4), color='red', linestyle='--', alpha=0.7, label='NASA CARA (1e-4)')
axes[0].axvline(x=np.log10(1e-5), color='orange', linestyle='--', alpha=0.7, label='Kelvins (1e-5)')
axes[0].axvline(x=np.log10(1e-6), color='green', linestyle='--', alpha=0.7, label='Starlink (1e-6)')
axes[0].legend(fontsize=9)

# CDF
sorted_pc = np.sort(df['pc'].values)
cdf = np.arange(1, len(sorted_pc) + 1) / len(sorted_pc)
axes[1].plot(sorted_pc, cdf, linewidth=2)
axes[1].set_xscale('log')
axes[1].set_xlabel('Pc')
axes[1].set_ylabel('Cumulative fraction')
axes[1].set_title('Pc Cumulative Distribution')
axes[1].axvline(x=1e-4, color='red', linestyle='--', alpha=0.5)
axes[1].axvline(x=1e-5, color='orange', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Summary thresholds
print(f'Pc >= 1e-4 (NASA CARA):   {(df["pc"] >= 1e-4).sum():>5} ({100*(df["pc"] >= 1e-4).mean():.1f}%)')
print(f'Pc >= 1e-5 (Kelvins):     {(df["pc"] >= 1e-5).sum():>5} ({100*(df["pc"] >= 1e-5).mean():.1f}%)')
print(f'Pc >= 1e-6 (Starlink):    {(df["pc"] >= 1e-6).sum():>5} ({100*(df["pc"] >= 1e-6).mean():.1f}%)')
print(f'Max Pc: {df["pc"].max():.6e}')
print(f'Median Pc: {df["pc"].median():.6e}')

In [ ]:
# Pc breakdown by object type pair
df['type_pair'] = df['sat1_type'] + ' vs ' + df['sat2_type']
type_pc = df.groupby('type_pair')['pc'].agg(['count', 'mean', 'median', 'max'])
type_pc = type_pc.sort_values('count', ascending=False)
print('Pc by object type pair:')
type_pc

## 3. Miss Distance Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of miss distance
axes[0].hist(df['miss_distance_km'], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0].set_xlabel('Miss Distance (km)')
axes[0].set_ylabel('Count')
axes[0].set_title('Miss Distance Distribution')

# Scatter: Pc vs miss distance
axes[1].scatter(df['miss_distance_km'], df['pc'], alpha=0.3, s=10, c='steelblue')
axes[1].set_yscale('log')
axes[1].set_xlabel('Miss Distance (km)')
axes[1].set_ylabel('Pc')
axes[1].set_title('Pc vs Miss Distance')
axes[1].axhline(y=1e-4, color='red', linestyle='--', alpha=0.5, label='NASA CARA')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Miss distance stats:')
print(f'  Min: {df["miss_distance_km"].min():.1f} km')
print(f'  Median: {df["miss_distance_km"].median():.1f} km')
print(f'  Mean: {df["miss_distance_km"].mean():.1f} km')
print(f'  Max: {df["miss_distance_km"].max():.1f} km')
print(f'  < 100 km: {(df["miss_distance_km"] < 100).sum()}')
print(f'  < 50 km:  {(df["miss_distance_km"] < 50).sum()}')
print(f'  < 25 km:  {(df["miss_distance_km"] < 25).sum()}')
print(f'  < 10 km:  {(df["miss_distance_km"] < 10).sum()}')

## 4. Object Analysis

In [ ]:
# Most common satellites involved in CDMs
sat_counter = Counter()
for _, row in df.iterrows():
    sat_counter[f"{row['sat1_name']} ({row['sat1_norad']})"] += 1
    sat_counter[f"{row['sat2_name']} ({row['sat2_norad']})"] += 1

top_sats = sat_counter.most_common(20)
names, counts = zip(*top_sats)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(range(len(names)), counts, color='steelblue', edgecolor='black')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Number of CDMs')
ax.set_title('Top 20 Satellites in CDMs')
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Object type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, sat_col in enumerate(['sat1_type', 'sat2_type']):
    vc = df[sat_col].value_counts()
    axes[i].pie(vc.values, labels=vc.index, autopct='%1.1f%%',
                colors=sns.color_palette('Set2', len(vc)))
    axes[i].set_title(f'Object 1 Types' if i == 0 else 'Object 2 Types')

plt.suptitle('Object Type Distribution in CDMs', fontsize=14)
plt.tight_layout()
plt.show()

# Type pair counts
print('\nType pair breakdown:')
print(df['type_pair'].value_counts().to_string())

## 5. Temporal Patterns

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# CDMs per day (by TCA date)
daily_tca = df.groupby(df['tca_dt'].dt.date).size()
axes[0].bar(range(len(daily_tca)), daily_tca.values, color='steelblue', edgecolor='black')
axes[0].set_xticks(range(0, len(daily_tca), max(1, len(daily_tca)//10)))
axes[0].set_xticklabels([str(d) for d in daily_tca.index[::max(1, len(daily_tca)//10)]], rotation=45, ha='right')
axes[0].set_xlabel('TCA Date')
axes[0].set_ylabel('Number of CDMs')
axes[0].set_title('CDMs per Day (by Time of Closest Approach)')

# CDMs per day (by creation date)
daily_created = df.groupby(df['created_dt'].dt.date).size()
axes[1].bar(range(len(daily_created)), daily_created.values, color='coral', edgecolor='black')
axes[1].set_xticks(range(0, len(daily_created), max(1, len(daily_created)//10)))
axes[1].set_xticklabels([str(d) for d in daily_created.index[::max(1, len(daily_created)//10)]], rotation=45, ha='right')
axes[1].set_xlabel('Creation Date')
axes[1].set_ylabel('Number of CDMs')
axes[1].set_title('CDMs per Day (by Creation Date)')

plt.tight_layout()
plt.show()

print(f'Average CDMs per day (TCA): {daily_tca.mean():.1f}')
print(f'Average CDMs per day (created): {daily_created.mean():.1f}')

## 6. Emergency Reportable Events

In [ ]:
emergency = df[df['emergency_reportable'] == 'Y'].copy()
print(f'Emergency reportable CDMs: {len(emergency)} / {len(df)} ({100*len(emergency)/len(df):.1f}%)')

if len(emergency) > 0:
    print(f'\nEmergency event statistics:')
    print(f'  Pc range: {emergency["pc"].min():.6e} to {emergency["pc"].max():.6e}')
    print(f'  Miss distance range: {emergency["miss_distance_km"].min():.1f} to {emergency["miss_distance_km"].max():.1f} km')
    print(f'  Unique pairs: {emergency.groupby(["sat1_norad", "sat2_norad"]).ngroups}')
    
    # Compare Pc distribution: emergency vs non-emergency
    fig, ax = plt.subplots(figsize=(12, 5))
    non_emerg = df[df['emergency_reportable'] != 'Y']
    ax.hist(np.log10(non_emerg['pc'].clip(lower=1e-20)), bins=40, alpha=0.6,
            label=f'Non-emergency (n={len(non_emerg)})', color='steelblue', edgecolor='black')
    ax.hist(np.log10(emergency['pc'].clip(lower=1e-20)), bins=40, alpha=0.6,
            label=f'Emergency (n={len(emergency)})', color='red', edgecolor='black')
    ax.set_xlabel('log10(Pc)')
    ax.set_ylabel('Count')
    ax.set_title('Pc Distribution: Emergency vs Non-Emergency CDMs')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No emergency reportable events in the dataset.')

## 7. RCS (Radar Cross Section) Analysis

In [ ]:
# RCS categories: SMALL, MEDIUM, LARGE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, rcs_col in enumerate(['sat1_rcs', 'sat2_rcs']):
    vc = df[rcs_col].value_counts()
    colors = {'SMALL': '#ff6b6b', 'MEDIUM': '#ffd93d', 'LARGE': '#6bcb77'}
    bar_colors = [colors.get(v, '#999999') for v in vc.index]
    axes[i].bar(vc.index, vc.values, color=bar_colors, edgecolor='black')
    axes[i].set_xlabel('RCS Category')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'Object {i+1} RCS Distribution')

plt.suptitle('Radar Cross Section Distribution', fontsize=14)
plt.tight_layout()
plt.show()

# RCS vs Pc
rcs_pc = df.groupby(['sat1_rcs', 'sat2_rcs'])['pc'].agg(['count', 'mean', 'max'])
print('\nPc by RCS pair:')
rcs_pc.sort_values('count', ascending=False)

## 8. Top 20 Highest-Risk Events

In [ ]:
top20 = df.nlargest(20, 'pc')[[
    'cdm_id', 'tca', 'pc', 'miss_distance_km',
    'sat1_name', 'sat1_type', 'sat2_name', 'sat2_type',
    'emergency_reportable'
]].copy()

top20['pc_fmt'] = top20['pc'].apply(lambda x: f'{x:.4e}')
top20['miss_km'] = top20['miss_distance_km'].round(1)
top20['tca_date'] = pd.to_datetime(top20['tca']).dt.strftime('%Y-%m-%d %H:%M')

display_cols = ['tca_date', 'pc_fmt', 'miss_km', 'sat1_name', 'sat1_type',
                'sat2_name', 'sat2_type', 'emergency_reportable']
print('Top 20 Highest Pc Events:')
top20[display_cols].to_string(index=False)

In [ ]:
# Visualize top events
fig, ax = plt.subplots(figsize=(14, 6))

labels = [f"{r['sat1_name']}\nvs {r['sat2_name']}" for _, r in top20.head(15).iterrows()]
pcs = top20.head(15)['pc'].values
colors = ['red' if r['emergency_reportable'] == 'Y' else 'steelblue'
          for _, r in top20.head(15).iterrows()]

bars = ax.barh(range(len(labels)), pcs, color=colors, edgecolor='black')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xscale('log')
ax.set_xlabel('Probability of Collision')
ax.set_title('Top 15 Highest-Risk CDM Events (red = emergency reportable)')
ax.axvline(x=1e-4, color='red', linestyle=':', alpha=0.5, label='NASA CARA threshold')
ax.legend()

plt.tight_layout()
plt.show()

## 9. Overlap with Our Active TLE Catalog

In [ ]:
# Load the most recent TLE snapshot
snapshots = sorted(TLE_SNAPSHOT_DIR.glob('*.json'))
if snapshots:
    latest = snapshots[-1]
    print(f'Latest TLE snapshot: {latest.name}')
    with open(latest) as f:
        tle_data = json.load(f)
    tle_norad_ids = {int(t.get('NORAD_CAT_ID', 0)) for t in tle_data if int(t.get('NORAD_CAT_ID', 0)) > 0}
    print(f'TLE catalog size: {len(tle_norad_ids)} objects')
    
    # CDM objects
    cdm_norad_ids = set(df['sat1_norad'].unique()) | set(df['sat2_norad'].unique())
    print(f'CDM unique objects: {len(cdm_norad_ids)}')
    
    overlap = cdm_norad_ids & tle_norad_ids
    cdm_only = cdm_norad_ids - tle_norad_ids
    print(f'\nOverlap: {len(overlap)} objects in both CDM and TLE catalog')
    print(f'CDM-only (not in our TLEs): {len(cdm_only)} objects')
    print(f'Coverage: {100*len(overlap)/len(cdm_norad_ids):.1f}% of CDM objects have TLEs')
    
    # What types are the CDM-only objects?
    cdm_only_types = Counter()
    for _, row in df.iterrows():
        if row['sat1_norad'] in cdm_only:
            cdm_only_types[row['sat1_type']] += 1
        if row['sat2_norad'] in cdm_only:
            cdm_only_types[row['sat2_type']] += 1
    print(f'\nObject types NOT in our TLE catalog:')
    for t, c in cdm_only_types.most_common():
        print(f'  {t}: {c}')
    
    # Venn-style bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(['In both', 'CDM only', 'TLE only'],
           [len(overlap), len(cdm_only), len(tle_norad_ids - cdm_norad_ids)],
           color=['#6bcb77', '#ff6b6b', '#4f8aff'], edgecolor='black')
    ax.set_ylabel('Number of Objects')
    ax.set_title('CDM vs TLE Catalog Overlap')
    plt.tight_layout()
    plt.show()
else:
    print('No TLE snapshots found. Run daily_predictions.py first.')

## 10. Training Data Quality: CDM Pc vs Model Predictions

Can our screening pipeline identify the same high-risk pairs that CDMs flag?
Compare CDM Pc ground truth against our heuristic risk scores from the daily pipeline.

In [ ]:
# Load the most recent prediction log
pred_logs = sorted((Path('../data/prediction_logs').glob('predictions_*.jsonl')))
if pred_logs:
    latest_pred = pred_logs[-1]
    print(f'Latest prediction log: {latest_pred.name}')
    preds = []
    with open(latest_pred) as f:
        for line in f:
            line = line.strip()
            if line:
                preds.append(json.loads(line))
    
    if preds:
        pred_df = pd.DataFrame(preds)
        print(f'Predictions: {len(pred_df)} pairs')
        
        # Build set of predicted pair NORAD IDs
        pred_pairs = set()
        for _, row in pred_df.iterrows():
            n1 = row.get('sat1_norad', 0)
            n2 = row.get('sat2_norad', 0)
            pred_pairs.add((min(n1, n2), max(n1, n2)))
        
        # Build set of CDM pair NORAD IDs
        cdm_pairs = set()
        for _, row in df.iterrows():
            n1 = row['sat1_norad']
            n2 = row['sat2_norad']
            cdm_pairs.add((min(n1, n2), max(n1, n2)))
        
        overlap_pairs = pred_pairs & cdm_pairs
        print(f'\nCDM pairs: {len(cdm_pairs)}')
        print(f'Predicted pairs: {len(pred_pairs)}')
        print(f'Overlap (our pipeline found CDM pair): {len(overlap_pairs)}')
        print(f'Detection rate: {100*len(overlap_pairs)/max(1,len(cdm_pairs)):.1f}%')
        
        # High-risk CDM pairs our pipeline missed
        high_pc_cdms = df[df['pc'] >= 1e-5]
        high_pc_pairs = set()
        for _, row in high_pc_cdms.iterrows():
            n1 = row['sat1_norad']
            n2 = row['sat2_norad']
            high_pc_pairs.add((min(n1, n2), max(n1, n2)))
        
        missed_high = high_pc_pairs - pred_pairs
        print(f'\nHigh-Pc (>=1e-5) pairs our pipeline missed: {len(missed_high)}/{len(high_pc_pairs)}')
    else:
        print('Prediction log is empty.')
else:
    print('No prediction logs found. Run daily_predictions.py first.')

## Summary

Key findings from the Space-Track CDM data:

1. **Data volume**: The rolling 30-day window provides a continuous stream of real conjunction data
2. **Pc distribution**: Most CDMs have Pc near the screening threshold (1e-7); a minority exceed NASA CARA maneuver threshold (1e-4)
3. **Miss distance**: Spans from single-digit km to thousands of km; not well correlated with Pc (covariance matters)
4. **Object types**: Debris is heavily represented (Fengyun 1C, Cosmos 2251 collisions still producing conjunctions)
5. **Catalog coverage**: Supplemental TLE fetch captures most CDM objects, but some debris/old payloads are missing
6. **Alert quality**: CDM-sourced alerts with real Pc values are a massive upgrade over TLE-screened heuristic scores

**Recommendation**: Use CDM Pc as the primary alert ranking signal, with TLE-screened pairs as supplementary coverage for objects not in CDMs.